# Graded Lab 3 — Validated Agent Pipelines
**Code Assignment 3 · CSI 4900 / CSI 5900 · Special Topics: Agentic AI · Oakland University**

Imagine asking an AI model to write a report from five records. What if some
records are missing information or come from sources you do not want to use?
You should check the records **before** asking the model to write.

In this lab, you will build a small program that does the following:

1. Make a four-step plan.
2. Get a list of records that we provide.
3. Use Python to check those records.
4. If the check passes, give the accepted records to a writer, then give its draft
   to an editor. If the check fails, stop without writing a report.

A **pipeline** is simply a sequence of steps where one step's result is used by
the next. Here, **validation** means checking whether the records meet a specific
rule. These two words explain the notebook title.

The planner, writer, and editor are three roles with different instructions. In
online mode, they can use the same AI model. Getting the records and checking them
are ordinary Python operations. This lab does not search the internet.

| Part | What you do | Graded? |
|---|---|---|
| A | Follow a complete example about a book-exchange event | No |
| B | Fill in missing values in three functions for a different event | **100 points** |
| C | Run the functions you completed together | No |
| D | See what happens when you make the check stricter | No |

**Estimated time: 50–65 minutes**, depending on your Python experience. You should
know functions, lists, dictionaries, and loops. The model-call pattern is the same
one used in Lab 2. The supporting code is provided.

| Exercise | What you fill in | Points |
|---|---|---:|
| 1 — `plan_brief` | Three values in a model request | 25 |
| 2 — `evaluate_sources` | A domain check, a count, a fraction, and a pass/fail check | 40 |
| 3 — `execute_plan` | Four function calls that pass the right information | 35 |

The tests check your Python functions, not the quality of an AI-written report.
They use different inputs, so your functions must work beyond the example records.
All graded work can be completed without an API key.

**Submit only `Graded_Lab_3_Validated_Pipelines.ipynb`, including your code and saved outputs.**


## Setup — start with the offline example

Use the step-by-step instructions in `README.md` if you need help opening Jupyter,
installing Python packages, or choosing a kernel. The **kernel** is the Python
process that runs your notebook cells. Keep the notebook, `lab_tools.py`,
`unittests.py`, and `requirements.txt` together in the extracted folder.

If you have not installed the packages yet, run this in a temporary notebook code
cell, then restart the kernel. Python 3.10 or newer is required.

```python
%pip install -r requirements.txt
```

Leave `RUN_LIVE = False` in the setup cell below. This selects **offline mode**:

| Step | Offline: `False` | Online: `True` |
|---|---|---|
| Plan, write, edit | Return text that we prepared in advance | Ask the Gemini model for text through `aisuite` |
| Get records | Return the provided local records | Return the same local records |
| Check records | Run Python calculations | Run the same Python calculations |

Offline replies let you practice without an API key. They are fixed examples,
not a model thinking about your input. Your Python functions still run normally.

**Run the next cell.** Its mode line should start with `OFFLINE`. A displayed model
name does not mean an online request was made. `lt` is a short name for the provided
`lab_tools` file; `unittests` contains the provided checks for your exercises.

After completing Part B, you may try online mode. Follow the README to get a
[Gemini key](https://aistudio.google.com/apikey), copy `.env.example` to `.env`, and
fill in `GEMINI_API_KEY`. Then change `RUN_LIVE` to `True` and restart/run all cells.
The helper uses `aisuite` to send requests to Google. In the default model name
`openai:gemini-3.5-flash-lite`, `openai:` selects a compatible request format; you
still use a **Gemini key**. The README explains how to change the model if needed.

Do not put a key in notebook cells or outputs. Do not purchase credits for this
assignment. If API access or quota prevents an online run, use offline mode and
add a short note. This does not reduce your programming grade.


In [ ]:
import json
import sys
from pathlib import Path
# The instructor notebook shares the student helpers.
if not Path("lab_tools.py").exists() and Path("../student/lab_tools.py").exists():
    sys.path.insert(0, str(Path("../student").resolve()))
import lab_tools as lt
import unittests
from lab_tools import (
    MODEL, PLAN_FORMAT, TOPIC, PREFERRED_DOMAINS, MIN_RATIO,
    get_client, parse_plan, validate_plan, hostname,
)

RUN_LIVE = False  # Change only when ready for the real API.
live_client = get_client() if RUN_LIVE else None
MODEL = lt.configured_model()
print("Mode:", "LIVE Gemini via aisuite" if RUN_LIVE else "OFFLINE: fixed replies + real Python execution")
print("Configured model:", MODEL)


# Part A — Worked example: a campus book exchange

**All code in Part A is complete. Read each explanation, then run the cell below it.**

We want to write a short report about a fictional book-exchange event. A *briefing*
here just means a short report. We have three records, each stored as a dictionary:

| Field | Meaning | Example |
|---|---|---|
| `id` | A label for citing the record | `BK-1` |
| `snippet` | A short piece of information about the event | The event exchanged 42 books. |

The third record deliberately has an empty `snippet`. Before writing, we require
at least **two out of every three records** to have some text in that field.
For this example, that means two of the three records must have text.

The complete program will make a plan, get the records, check them, and then write
and edit a report. If the check fails, it stops before writing.

### A1. Store the topic and the three records

The next cell only stores data. It does not call a model or check the records.
All event details are invented for this exercise.


In [ ]:
BOOK_TOPIC = "Prepare a campus book-exchange briefing."
book_records = [
    {"id": "BK-1", "snippet": "The fictional pilot exchanged 42 books during a two-hour session."},
    {"id": "BK-2", "snippet": "The fictional team has four listed volunteers; attendance is unconfirmed."},
    {"id": "BK-3", "snippet": ""},
]


### A2. Define a function that asks for a plan

`def` defines a function; it does not run its body yet. The function below receives
the topic, a `client` that supplies replies, and a model name. Its prompt asks for
four tasks, in this exact order:

| Label | Task |
|---|---|
| `[SEARCH]` | Get the provided records |
| `[VALIDATE]` | Check whether enough records have text |
| `[WRITE]` | Write a draft only if the check passes |
| `[EDIT]` | Check the draft against the records |

The order is supplied; the planner writes the task descriptions. A plan is just a
to-do list. Writing “get records” in the list does not get the records yet.

Inside the function, `client.chat.completions.create(...)` requests a reply.
`messages` contains our prompt, `model` selects the model, and `temperature=0.2`
sets how much the model's output may vary. `PLAN_FORMAT` asks for JSON containing
a `steps` list. JSON is a text format that lets Python read the result reliably.
`response.choices[0].message.content` gets the reply text; `parse_plan` turns it
into a Python list and checks the labels and order.

The same request line works with a real client or our offline simulator.
**Running this definition cell does not make a request. We call it in A5.**


In [ ]:
def plan_book_exchange(topic, client, model=MODEL):
    prompt = f"""Plan this briefing: {topic}
    Return only JSON with a steps list of four strings in order:
    [SEARCH] retrieve local records; [VALIDATE] check snippet completeness;
    [WRITE] draft only if the gate passes; [EDIT] check the draft against evidence.
    Each tag must be followed by a space and a task description."""
    response = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        temperature=0.2, response_format=PLAN_FORMAT,
    )
    return parse_plan(response.choices[0].message.content)


### A3. Define a function that checks the records

This second function is ordinary Python. It makes no model request. It receives
records and a required fraction, `min_ratio`, whose default is `2/3`.

Read its variables as follows:

| Variable | Meaning for the three book records |
|---|---|
| `complete` | Records whose snippet contains text: BK-1 and BK-2 |
| `total` | All three records |
| `ratio` | Two with text divided by three total |
| `passed` | `True`: the records are not empty and the fraction reaches 2/3 |
| `approved_ids` | The labels `BK-1` and `BK-2` |

In the list comprehension, `r` is one record. `r.get("snippet", "")` reads its
snippet, or uses an empty string if the field is missing. `.strip()` removes
surrounding spaces. An empty string counts as false in the `if`, so a blank or
spaces-only snippet is excluded. `len(...)` counts items in a list.

`... if total else 0.0` prevents division by zero when there are no records.
`>=` includes equality: reaching exactly 2/3 is enough. The function returns a
dictionary describing the check. **This cell defines it; A7 will call it.**


In [ ]:
def evaluate_book_records(records, min_ratio=2/3):
    complete = [r for r in records if r.get("snippet", "").strip()]
    total = len(records)
    ratio = len(complete) / total if total else 0.0
    return {"total": total, "approved": len(complete), "ratio": ratio,
            "passed": total > 0 and ratio >= min_ratio,
            "approved_ids": [r["id"] for r in complete]}


### A4. Choose where the replies will come from

To make this example work without an API key, we prepare three text replies:
a plan, a draft, and an edited report. `json.dumps(...)` turns the plan dictionary
into JSON text, like the text a model would return.

`book_client = live_client if RUN_LIVE else ...` selects one of two options:

- `RUN_LIVE=True`: use the real client. These prepared replies are not used.
- `RUN_LIVE=False`: use `ScriptedClient`, our offline simulator. Each request
  takes the next reply from the list: plan first, draft second, edited report third.

The simulator does not reason about the prompt. It just returns prepared text in
order. That is why you can see the draft and edited report written in the code.
The two adjacent strings at the end join into **one** edited-report reply.

**This cell prepares the client. No replies have been requested yet.**


In [ ]:
book_plan_text = json.dumps({"steps": [
    "[SEARCH] Retrieve the book-exchange records.",
    "[VALIDATE] Check the share with nonempty snippets.",
    "[WRITE] Draft from complete records if the gate passes.",
    "[EDIT] Check the draft and preserve the attendance uncertainty.",
]})
book_client = live_client if RUN_LIVE else lt.ScriptedClient([
    book_plan_text,
    "The fictional pilot exchanged 42 books in two hours [BK-1]. Four volunteers are listed [BK-2].",
    "The fictional pilot exchanged 42 books in two hours [BK-1]. Four volunteers are listed, "
    "but attendance is not confirmed [BK-2]. The records do not establish future demand.",
])


### A5. Call the planner and print its plan

Now we call `plan_book_exchange`. In offline mode, the simulator returns its first
reply. `parse_plan` converts that reply to a list, which is stored in `book_plan`.
`json.dumps(..., indent=2)` displays the list with indentation for easier reading.

**Expected output:** four strings starting with `[SEARCH]`, `[VALIDATE]`, `[WRITE]`,
and `[EDIT]`. At this point, no records have been checked and no report has been
written. The remaining two offline replies are waiting for the writer and editor.


In [ ]:
book_plan = plan_book_exchange(BOOK_TOPIC, book_client)
print(json.dumps(book_plan, indent=2))


### A6. Define the function that runs the four steps

You now have a plan, but none of its tasks has been carried out. The next function,
`run_book_pipeline`, carries them out in order.

Its `workers` argument is a **dictionary containing functions**. Looking up
`workers["write"]` gives you the writing function. Adding parentheses calls it.
For example, `workers["write"](topic, task, approved)` calls that function with
the topic, the writing instructions, and the accepted records.

| Function in `workers` | Information it receives | What it returns |
|---|---|---|
| `search` | Topic and search task | A list of records |
| `write` | Topic, writing task, and accepted records | A draft as a string |
| `edit` | Topic, editing task, the actual draft, and accepted records | The final report as a string |

Read the next function in this order:

1. `validate_plan(plan)` checks the four labels and their order.
2. `tasks` removes the labels. For example, splitting
   `"[SEARCH] Retrieve records"` at its first space gives
   `["[SEARCH]", "Retrieve records"]`; `[1]` selects the task text.
3. `records` receives the search function's result. `evaluation` receives the
   result of checking those records. This is where the record check actually runs.
4. `history` starts a log of the search and check results. It is a Python list
   for us to inspect, not a conversation that models automatically share.
5. If `passed` is `False`, `return` ends the function immediately. The status is
   `"blocked"` and `report` is `None`, meaning there is no report.
6. Otherwise, `approved` keeps only records whose IDs passed the check. The writer
   receives those records. The editor receives the writer's returned `draft` **and**
   those same records. The status is then `"complete"`.

The second plan step describes the check, but Python determines its actual rule:
`evaluate_book_records` checks for nonempty snippets. The plan text cannot change
that rule. Python uses predefined functions; it does not execute the plan as code.

**Run this cell to define the function. The function will be called in A7.**


In [ ]:
def run_book_pipeline(plan, topic, workers, min_ratio=2/3):
    validate_plan(plan)
    tasks = [step.split(" ", 1)[1] for step in plan]
    records = workers["search"](topic, tasks[0])
    evaluation = evaluate_book_records(records, min_ratio)
    history = [
        {"agent": "search", "step": plan[0], "output": records},
        {"agent": "validate", "step": plan[1], "output": evaluation},
    ]
    if not evaluation["passed"]:
        return {"status": "blocked", "history": history, "records": records,
                "evaluation": evaluation, "report": None}
    approved = [r for r in records if r["id"] in evaluation["approved_ids"]]
    draft = workers["write"](topic, tasks[2], approved)
    report = workers["edit"](topic, tasks[3], draft, approved)
    history += [
        {"agent": "write", "step": plan[2], "output": draft},
        {"agent": "edit", "step": plan[3], "output": report},
    ]
    return {"status": "complete", "history": history, "records": records,
            "evaluation": evaluation, "report": report}


### A7. Create the worker functions and run the example

`make_workers` supplies the search, writing, and editing functions. We then replace
its search function with one that returns copies of the three book records.
`lambda topic, task: ...` is a short way to define a function. It accepts the two
arguments expected by search, but this example always returns `book_records`.
`dict(r)` makes a copy of each record dictionary. No web search takes place.

The writer and editor use different prompts. Sharing one client does **not** give
them shared memory. That is why the editor must receive the `draft` and accepted
records explicitly. In offline mode, their replies still come from our fixed list.

This cell calls `run_book_pipeline` for the first time. The check finds two records
with text out of three, so it passes. The writer receives BK-1 and BK-2, and the
editor receives the resulting draft plus those records. `book_run` stores the
results; `show_run` displays them without making another model request.

**Expected output:** `STATUS: complete`, four executed steps, the check results,
and a short report. Do not change any code here.


In [ ]:
book_workers = lt.make_workers(book_client, model=MODEL)
book_workers["search"] = lambda topic, task: [dict(r) for r in book_records]
book_run = run_book_pipeline(book_plan, BOOK_TOPIC, book_workers)
lt.show_run(book_run)


### A8. Try the same records with a stricter check

Now call the same function with `min_ratio=0.9`, meaning 90% of records must have
text. Only 2/3 have text, so the function stops at the failed check.

This call reuses the plan; it does not call the planner again. Search and the
Python check run again, but writing and editing do not run. It therefore consumes
no new model calls and no more replies from the offline simulator.


In [ ]:
# A stricter gate fails: writer and editor are never invoked. No new model calls.
blocked_book = run_book_pipeline(book_plan, BOOK_TOPIC, book_workers, min_ratio=0.9)
print("Stricter gate:", blocked_book["status"], "| executed stages:",
      [row["agent"] for row in blocked_book["history"]])


### What the two runs show

The first run should display `STATUS: complete` and a report. The second should show:

```text
Stricter gate: blocked | executed stages: ['search', 'validate']
```

The word **gate** in this output means the pass/fail check that controls whether
writing may start. Here is why the two runs differ:

| Run | Records with text | Required fraction | Result |
|---|---|---|---|
| First | 2 of 3 | At least 2/3 | Continue to writing and editing |
| Second | 2 of 3 | At least 0.9, or 90% | Stop after checking |

This check only asks whether a snippet has text. Text can be present and still be
wrong. Do not interpret a passing result as proof that a record is factually correct.

We checked **one part** of the program: the records returned by search. This is
called *component-level evaluation*. A component is one part of a larger program.
Notice that we could make the check stricter without asking for another report.


# Part B — Your task: a campus repair-cafe report

**Complete only the three functions marked as exercises. The rest is provided.**

A repair cafe is an event where volunteers try to repair visitors' belongings.
For this fictional campus event, we provide five records about a previous event,
volunteers, and intake instructions. Your program will use them to prepare a report.

The sequence is the same as Part A: make a plan, get records, check them, write,
and edit. **The rule for checking records is different.**

| Part A | Part B |
|---|---|
| Does each record have a nonempty snippet? | Does each record's URL belong to a domain on our preferred list? |
| Require at least 2/3 to pass | By default, require at least 0.6 (60%) to pass |

A **domain** identifies a website, such as `campus.example`. Our preferred list
says which domains count as acceptable **for this exercise**. Each record includes
a URL so your code can make that decision. The `.example` addresses are fictional;
do not try to visit them. The search helper returns the same five records each time.

Your check will divide the number of accepted records by the number of **all**
records returned. Keep rejected and invalid URLs in that total. If the result
passes the required fraction, only accepted records may be used for writing.

**Run the next cell to inspect the five records and the preferred-domain list.**
There is nothing to fill in in this cell. A preferred domain does not guarantee
that its content is correct, up to date, or useful for the topic.


In [ ]:
print("Topic:", TOPIC)
print("Preferred domains:", sorted(PREFERRED_DOMAINS))
print("Threshold:", MIN_RATIO)
for record in lt.search_records(TOPIC, "Retrieve local records"):
    print(record["id"], "|", record["url"], "|", record["snippet"])


## Exercise 1 — Request a plan (`plan_brief`, 25 points)

**Your goal:** send the supplied prompt to the supplied client, then return the
four-step plan in its reply. Compare the model request in Part A's
`plan_book_exchange` function if you need an example.

Replace the three `...` values inside the marked region:

| Argument | What to pass |
|---|---|
| `model` | The function's `model` argument |
| `messages` | A list containing one user message whose content is `prompt` |
| `temperature` | `0.2` |

The prompt is already written. Keep `response_format=PLAN_FORMAT` and the
`parse_plan(...)` return line: together they request and read the four-step format.
This function returns a list of task strings, not a finished report.

Keep `client = client or get_client()`. It uses the client passed into the function
when one is available; otherwise it creates the real API client. The tests pass in
an offline client that records requests and returns prepared replies. Creating a
different client inside your answer would prevent the tests from using it.

**After editing:** run the function cell, then its test cell. The tests check that
you make one request, send the current topic and model, keep the requested settings,
and return the plan from the reply. They change the topic and reply between cases,
so do not return the fixed book-exchange plan. Format errors are handled by the
provided parser; you do not need to write another parser.


In [ ]:
def plan_brief(topic, client=None, model=MODEL):
    """Return four tagged task strings from one model response."""
    client = client or get_client()
    prompt = f"""Plan a short evidence-based briefing about: {topic}
    Return ONLY JSON with a steps list containing exactly four strings, in order:
    [SEARCH] a task to retrieve the supplied local records for the topic;
    [VALIDATE] a task to check the preferred-source ratio before drafting;
    [WRITE] a task to draft using only approved records if validation passes;
    [EDIT] a task to check the draft against those records and preserve uncertainty.
    Each string begins with its tag followed by a space and a meaningful task.
    Do not claim that preferred domains guarantee truth. Do not browse the web.
    """
    ### START CODE HERE ###
    # Fill three values: the supplied model, the user prompt, and temperature 0.2.
    response = client.chat.completions.create(
        model=...,
        messages=...,
        temperature=..., response_format=PLAN_FORMAT,
    )
    ### END CODE HERE ###
    return parse_plan(response.choices[0].message.content)


In [ ]:
unittests.test_plan_brief(plan_brief)


## Exercise 2 — Check the sources (`evaluate_sources`, 40 points)

**Your goal:** decide which records have an acceptable URL, then calculate whether
enough records passed. This function uses Python only; it makes no model request.

The function receives:

| Argument | Meaning |
|---|---|
| `records` | A list of record dictionaries; each has its own unique string `id` |
| `preferred_domains` | A set of acceptable domain names, such as `{"archive.example"}` |
| `min_ratio` | The smallest fraction that passes, from `0` to `1`; default `0.6` |

The preferred domain names are already lowercase, with no `https://`, path, or
trailing dot. Record URLs may be missing or invalid. The provided `hostname`
helper extracts the host from a URL and makes it lowercase. For example,
`https://docs.archive.example/page` has host `docs.archive.example`. The helper
returns `""` for an invalid or non-web URL. You do not need to write URL parsing.

### Which hosts count as acceptable?

A nonempty host must either **equal** a preferred domain or end with a **dot
followed by** that domain. This allows a subdomain such as `docs.archive.example`.
Merely finding the same letters somewhere in a host is not enough.

| Preferred domain | Host from a record's URL | Accept? |
|---|---|---|
| `archive.example` | `archive.example` | Yes — exact match |
| `archive.example` | `docs.archive.example` | Yes — subdomain |
| `archive.example` | `notarchive.example` | No — missing the dot boundary |
| `archive.example` | `archive.example.other.example` | No — ends in a different domain |

### What to fill in

There are four `...` expressions in two marked regions:

1. `preferred`: `True` when this record's nonempty host matches **any** domain
   using the rule above; otherwise `False`.
2. `approved`: the number of IDs collected in `approved_ids`.
3. `ratio`: accepted records divided by **all** records; use `0.0` when there are none.
4. `passed`: `True` only when there is at least one record and `ratio` is greater
   than **or equal to** `min_ratio`. Otherwise `False`.

The loop, collection of accepted IDs, and return dictionary are already written.
`any(...)`, `len(...)`, and `.endswith(...)` may be useful. Do not change the input
records or the preferred-domain set.

### Example of the expected result

If 2 of 4 records have acceptable URLs and the required fraction is `0.5`, return
counts of 4 and 2, a ratio of `0.5`, and `passed=True`. The `approved_ids` list must
contain those two records' IDs in their original order. Equality passes.

Keep these edge cases in mind:

- A rejected, missing, or invalid URL still counts toward `total`.
- An empty record list always fails, even when `min_ratio` is zero.
- A nonempty list with threshold zero follows the ordinary fraction rule.
- Count records, not distinct websites. You may assume record IDs are unique.
- An empty preferred-domain set means no records have an acceptable domain.

**After editing:** rerun the function cell and then the test cell. Tests include
different domain lists, misleading host names, invalid URLs, and exact threshold
matches. They check the returned counts, ratio, Boolean result, and IDs.


In [ ]:
def evaluate_sources(records, preferred_domains, min_ratio=0.6):
    """Return counts, ratio, pass/fail, and approved IDs without modifying inputs."""
    if not 0 <= min_ratio <= 1:
        raise ValueError("min_ratio must be between 0 and 1.")
    approved_ids = []
    for record in records:
        host = hostname(record.get("url", ""))
        ### START CODE HERE ###
        # Match an exact domain OR a real subdomain (dot boundary).
        preferred = ...
        ### END CODE HERE ###
        if preferred:
            approved_ids.append(record["id"])
    total = len(records)
    ### START CODE HERE ###
    approved = ...
    ratio = ...
    passed = ...
    ### END CODE HERE ###
    return {"total": total, "approved": approved, "ratio": ratio,
            "passed": passed, "approved_ids": approved_ids}


In [ ]:
unittests.test_evaluate_sources(evaluate_sources)


## Exercise 3 — Run the steps (`execute_plan`, 35 points)

**Your goal:** connect the supplied functions by passing each one the information
it needs. Complete Exercise 2 first, because this function calls your
`evaluate_sources` function.

You receive a plan, topic, preferred-domain set, minimum fraction, and a `workers`
dictionary. As in Part A, `workers` stores the search, write, and edit functions.
The provided loop handles one plan step at a time. For example:

```text
step: [WRITE] Draft the event report.
task: Draft the event report.
```

`step` includes the label; `task` is the instruction after it. Fill the four
marked calls using this guide:

| Save the result in | Function to call | Arguments, in order |
|---|---|---|
| `records` | The search function in `workers` | `topic`, `task` |
| `evaluation` | Your `evaluate_sources` | `records`, `preferred_domains`, `min_ratio` |
| `draft` | The write function in `workers` | `topic`, `task`, `approved_records` |
| `report` | The edit function in `workers` | `topic`, `task`, `draft`, `approved_records` |

Use the records returned **during this run**, not a copy of the sample written
into your answer. Give the editor the draft actually returned by the writer.

The rest is already provided. Do not add another loop, retry, or model client:

- An invalid plan is rejected before any worker runs.
- If the source check fails, the function stops with `status="blocked"`,
  `report=None`, and two history entries: search and validate.
- If it passes, `approved_records` keeps only accepted records. Writing and editing
  run, and the function returns `status="complete"` with four history entries.
- `history` records each step and its returned result so you can inspect the run.

**After editing:** rerun this function cell, then the test cell. The tests supply
small replacement worker functions that record what you pass to them. They check
the actual records, settings, accepted evidence, draft, final report, and order of
calls. In a failed run, neither writing function should be called.


In [ ]:
def execute_plan(plan, topic, preferred_domains, min_ratio=0.6, workers=None):
    """Run a validated four-stage plan; a failed evaluation blocks report creation."""
    validate_plan(plan)
    if workers is None:
        raise ValueError("Supply the provided worker dictionary.")
    history = []
    records, evaluation, approved_records = [], None, []
    draft, report = None, None
    for step in plan:
        tag, task = step.split(" ", 1)
        if tag == "[SEARCH]":
            ### START CODE HERE ###
            records = ...
            ### END CODE HERE ###
            output = records
            agent = "search"
        elif tag == "[VALIDATE]":
            ### START CODE HERE ###
            evaluation = ...
            ### END CODE HERE ###
            output = evaluation
            agent = "validate"
            history.append({"agent": agent, "step": step, "output": output})
            if not evaluation["passed"]:
                return {"status": "blocked", "history": history, "records": records,
                        "evaluation": evaluation, "report": None}
            approved_records = [r for r in records if r["id"] in evaluation["approved_ids"]]
            continue
        elif tag == "[WRITE]":
            ### START CODE HERE ###
            draft = ...
            ### END CODE HERE ###
            output = draft
            agent = "write"
        else:  # validate_plan has already restricted the final role to [EDIT].
            ### START CODE HERE ###
            report = ...
            ### END CODE HERE ###
            output = report
            agent = "edit"
        history.append({"agent": agent, "step": step, "output": output})
    return {"status": "complete", "history": history, "records": records,
            "evaluation": evaluation, "report": report}


In [ ]:
unittests.test_execute_plan(execute_plan)


## Check all three exercises

Run the next cell after completing the exercises. It prints a result and points
for each check. Your target is `TOTAL: 100/100`.

If a check fails, read its message, edit the relevant function, **run that function's
definition cell again**, and then rerun the tests. Editing a cell without running
it does not update the function stored in the kernel.

The untouched starter normally earns **15/100**, because some checks test code
we supplied. Failed checks and `Ellipsis` messages are expected while `...` remains
in your answers; they do not mean your environment is broken.

Passing all tests means your functions met the tested requirements. It does not
prove that a live model's report is accurate. Your instructor also reviews the
code and reruns the three functions with the original helper files. Write general
solutions that use their inputs, not answers tailored to individual test cases.


In [ ]:
grade = unittests.run_all_tests(plan_brief, evaluate_sources, execute_plan)


# Part C — Use your functions to make the repair-cafe report

**Run this cell after Part B shows 100/100.** Otherwise, it prints a message
explaining that Part C was skipped.

The cell uses your functions to make the plan, get the five local records, check
their domains, and send accepted records to the writer and editor. It displays
the result and a list of the steps that ran. You do not fill in anything here.

With the supplied records and the default threshold, expect:

- `STATUS: complete`.
- Three accepted records out of five: `RC-11`, `RC-12`, and `RC-13`.
- A report and four history entries: search, validate, write, and edit.

In offline mode, the report text was prepared in advance. The line `Model calls
in Part C: 3` counts three requests to the **offline simulator**; it does not mean
three requests were sent to Google. Your record checks and function calls still run.

In online mode, planning, writing, and editing each call Gemini through `aisuite`.
A successful Part C takes three API calls. Part A takes another three, so running
both successfully takes six. Search, validation, and graded tests make no API calls.
The helper waits at least five seconds between attempts and does not retry
automatically. If an API request fails, it shows an error; switch back to offline
mode if you cannot resolve the access or quota issue.

When reading the report, look for record IDs next to claims and check whether
uncertainty was preserved. A record coming from a preferred domain is not enough
to prove every sentence in the report is correct. This observation is not graded.


In [ ]:
repair_run = None
if grade["earned"] == 100:
    repair_client = live_client if RUN_LIVE else lt.offline_client(TOPIC)
    before_calls = repair_client.n_calls
    repair_plan = plan_brief(TOPIC, client=repair_client, model=MODEL)
    repair_workers = lt.make_workers(repair_client, model=MODEL)
    repair_run = execute_plan(repair_plan, TOPIC, PREFERRED_DOMAINS,
                              min_ratio=MIN_RATIO, workers=repair_workers)
    lt.show_run(repair_run)
    print("Model calls in Part C:", repair_client.n_calls - before_calls)
else:
    print("Part C skipped: complete the three exercises and pass the public checks first.")


# Part D — Make the required fraction stricter

**Optional: run and observe. No code to fill in, no written response to submit,
and no new model calls.**

The next cell checks the same records twice. It changes only the minimum fraction
needed to pass:

| Minimum fraction | Actual result | What should happen? |
|---|---|---|
| `0.6` (60%) | 3 of 5 = 60% | Pass |
| `0.8` (80%) | 3 of 5 = 60% | Fail |

It then runs your pipeline at `0.8`. The result should be `blocked`, with only
search and validate in the history. The provided `must_not_write` function raises
an error if the program mistakenly tries to write or edit after the failed check.

Notice that you can check these records without paying for another report.
That is the benefit of checking one component separately.

Something to think about: if you deleted the two rejected records **before**
calculating the fraction, you would report 3/3 instead of 3/5. That would hide the
problem in the search results. This is why Exercise 2 counts every returned record
in the total, even when its URL is invalid or unapproved.


In [ ]:
if repair_run is not None:
    for threshold in (0.6, 0.8):
        result = evaluate_sources(repair_run["records"], PREFERRED_DOMAINS, threshold)
        print(f"threshold={threshold:.1f}: {result['approved']}/{result['total']} "
              f"= {result['ratio']:.0%}, passed={result['passed']}")

    def must_not_write(*args):
        raise AssertionError("A blocked run must not invoke a writing role.")

    strict_workers = {"search": lt.search_records, "write": must_not_write, "edit": must_not_write}
    strict_run = execute_plan(repair_plan, TOPIC, PREFERRED_DOMAINS,
                              min_ratio=0.8, workers=strict_workers)
    print("Stricter pipeline:", strict_run["status"],
          "| stages:", [row["agent"] for row in strict_run["history"]])
else:
    print("Complete Part B and run Part C to inspect the gate.")


## Save and submit

1. Finish the three exercises and rerun their tests.
2. Restart the kernel and run every cell from the top. This checks that your
   notebook does not depend on variables left over from an earlier run.
3. Keep the outputs and save the notebook. Online mode is optional when free API
   access is available. If access or quota prevents it, use `RUN_LIVE=False` and
   add a short Markdown note. You do not lose programming points for this.
4. Submit **only `Graded_Lab_3_Validated_Pipelines.ipynb`**. Do not submit `.env`,
   helper files, a ZIP, screenshots, HTML, or a separate report.

Only edit marked exercise regions and `RUN_LIVE`, apart from the optional access
note. Remove any temporary setup or diagnostic cells you added from the README.
Your solution must work with the original helpers and the three function
definitions; extra cells cannot supply missing parts of a graded answer.

## If something goes wrong

| What you see | What to try |
|---|---|
| `Ellipsis` or errors involving `...` | Fill the remaining exercise blanks, then rerun the function definition and tests. |
| Exercise 3 fails after you edited Exercise 2 | Run the Exercise 2 definition cell again before testing Exercise 3. |
| `NameError` or an old result | Restart the kernel and run from the top. |
| Cannot import `lab_tools` | Extract the whole ZIP. Keep both helper files beside the notebook. |
| Cannot import `aisuite` or `dotenv` | Follow the README to install requirements in the kernel you selected. |
| A plan or JSON error during an online run | Keep the supplied format/parser. Retry manually if quota allows; contact your TA if it persists. |
| API error 401/403 | Check the key and project access. Do not print or share the key. |
| API error 404 | Ask your TA which accessible model to use, update `.env`, and restart the kernel. |
| API error 429 | Check the project's quota and wait for the relevant reset. Use offline mode meanwhile. |

See `README.md` for the full setup instructions. Contact your TA if you need help.
Technical references: [AISuite](https://github.com/andrewyng/aisuite) and
[Google's compatibility guide](https://ai.google.dev/gemini-api/docs/openai).
